# Imports

In [ ]:
import pandas as pd
import plip_analysis as pa
from pathlib import Path
import plotly.express as px
import json

# Load the data

In [ ]:
crystal_interactions = {csv_path.stem: pa.PLIntReport.from_csv(csv_path) for csv_path in Path("mers_crystal_interactions").glob("*.csv")}
docked_interactions = {csv_path.stem: pa.PLIntReport.from_csv(csv_path) for csv_path in Path("mers_docked_interactions").glob("*.csv")}

In [ ]:
new_crystal_interactions = {}
for key, value in crystal_interactions.items():
    crystal_name = "-".join(key.split("_")[2].split("-")[:2])
    new_crystal_interactions[crystal_name] = value

In [ ]:
score_list = []
missing = []
for level in pa.FingerprintLevel:
    for name, docked_plint_report in docked_interactions.items():
        common_key = name.split("_")[0]
        crystal_plint_report = new_crystal_interactions.get(common_key)
        if crystal_plint_report is None:
            missing.append(name)
            continue
        score_list.append({'ASAP_Ligand_ID': common_key, 'Variant': 'MERS', **pa.InteractionScore.from_fingerprints(crystal_plint_report, docked_plint_report, level).dict()})

In [ ]:
df = pd.DataFrame.from_records(score_list)

In [ ]:
df

# Get cumulative distribution 

The ratio of the intersection is the tversky index or recall

In [ ]:
df["ratio_of_intersection"] = df["number_of_interactions_in_intersection"] / df["number_of_interactions_in_reference"]

the ratio of the query to intersection is also interesting 

In [ ]:
df['ratio_of_query'] = df["number_of_interactions_in_query"] / df["number_of_interactions_in_reference"]

In [ ]:
by_interaction_type = df[df["provenance"] == pa.FingerprintLevel.ByInteractionType.value]
by_everything = df[df["provenance"] == pa.FingerprintLevel.ByEverything.value]

In [ ]:
by_everything.reference_fingerprint[200]

In [ ]:
fig = px.ecdf(df, x="tversky_index", height=400, width=800, color='provenance', template='simple_white')
fig.update_layout(legend={'title': 'Similarity Metric'})
fig.update_yaxes(title='Cumulative Probability')
fig.update_xaxes(title='PLIF Recall')
fig.show()
fig.write_image("cdf_fingerprint_comparison.png")

In [ ]:
import plotly.graph_objects as go
import numpy as np
# Extract the data
data = by_interaction_type["tversky_index"]

# Create the histogram
fig = go.Figure()
fig.add_trace(go.Histogram(x=data, name='Histogram', histnorm='probability'))

# # Calculate the CDF
hist, bins = np.histogram(data, bins=30)
cdf = np.cumsum(hist) / len(data)
# 
# Create the CDF trace
fig.add_trace(go.Scatter(x=bins, y=cdf, name='CDF', mode='lines'))

# Update layout
fig.update_layout(title='PLIF Recall for MERS Predictions', height=400, width=600, template="simple_white")

# Update axis titles
fig.update_xaxes(title="PLIF Recall")
fig.update_yaxes(title="Probability")

# Show the plot
fig.show()

In [ ]:
fig.write_image("plif_recall_for_mers_predictions.png")

# Generate Stacked Bar Chart

In [ ]:
key, value = crystal_interactions.popitem()

In [ ]:
import json
jsonable_dict = json.loads(value.json())

In [ ]:
score_list = []
records = []
missing = []
for name, docked_plint_report in docked_interactions.items():
    common_key = name.split("_")[0]
    crystal_plint_report = new_crystal_interactions.get(common_key)
    if crystal_plint_report is None:
        missing.append(name)
        continue
    for interaction_list, name in [(crystal_plint_report.interactions, "Crystal"), (docked_plint_report.interactions, "Docked")]:
        for interaction in interaction_list:
            record_dict = json.loads(interaction.json())
            record_dict.update({"Structure": common_key, "StructureType": name})
            records.append(record_dict)

In [ ]:
tidy_df = pd.DataFrame.from_records(records)

In [ ]:
tidy_df['fingerprint'] = tidy_df['interaction_type'] + "_" + tidy_df['protein_residue_type'] + "_" + tidy_df['protein_residue_number'].astype(str) + "_" + tidy_df['to_sidechain'].apply(lambda x: 'SC' if x else "BB") + "_" + tidy_df["ligand_atom_type"] + "_" + tidy_df['protein_atom_type']

In [ ]:
tidy_df = tidy_df.drop(columns=[col for col in tidy_df.columns if not col in ['StructureType', "Structure", "interaction_type", 'fingerprint', 'count']])

In [ ]:
crystal_df = tidy_df[tidy_df["StructureType"] == "Crystal"]

In [ ]:
groupeddf = tidy_df.groupby(["StructureType", "Structure", "interaction_type"])["count"].sum().reset_index()

In [ ]:
crystal_df = tidy_df[tidy_df["StructureType"] == "Crystal"]
crystal_records = crystal_df.to_dict(orient="Records")

docked_df = tidy_df[tidy_df["StructureType"] == "Docked"]
docked_records = docked_df.to_dict(orient="Records")

In [ ]:
recall = pd.merge(crystal_df.drop(columns='StructureType'), docked_df.drop(columns='StructureType'), how='left',on=['Structure', 'fingerprint'], suffixes=['_Crystal', '_Docked']).fillna(0)

In [ ]:
recalls = recall.groupby(['interaction_type_Crystal', 'Structure'])['count_Docked'].mean().groupby('interaction_type_Crystal').describe().reset_index()
recalls['Category'] = 'Recall'
recalls.columns = [col if not col == 'interaction_type_Crystal' else 'interaction_type' for col in recalls.columns]

In [ ]:
totals = docked_df.groupby(['interaction_type', 'Structure'])['count'].sum() / crystal_df.groupby(['interaction_type', 'Structure'])['count'].sum()
totals = totals.fillna(0).groupby('interaction_type').describe().reset_index()

In [ ]:
totals['Category'] = 'Calculated'

In [ ]:
bar_chart_df = pd.concat([recalls, totals])

In [ ]:
bar_chart_df

In [ ]:
docked_df[docked_df['interaction_type'] == 'SaltBridge']

In [ ]:
crystal_df[crystal_df['interaction_type'] == 'SaltBridge']

In [ ]:
fig = px.bar(bar_chart_df, x="interaction_type", y="mean", color='Category', barmode='group', height=400, width=600, template='simple_white')
fig.update_xaxes(title="Interaction Type")
fig.update_yaxes(title="Recall")
fig.show()

In [ ]:
fig.write_image("by_interaction_type_bar_chart.png")

The pipeline should be:
1) calculate interaction fingerprint
2) for each interaction type, count how many are recalled for each structure and divide by number of interactions in original structure (tversky)
3) take the average +/- sd of the recall

In [ ]:
pa.calculate_fingerprint()

In [ ]:
records = groupeddf.to_dict(orient="Records")

In [ ]:
results = []
for structure in groupeddf.Structure.unique():
    for interaction_type in groupeddf.interaction_type.unique():
        docked = [rcd for rcd in docked_records if rcd["Structure"] == structure and rcd["interaction_type"] == interaction_type]
        crystal = [rcd for rcd in crystal_records if rcd["Structure"] == structure and rcd["interaction_type"] == interaction_type]
        
        results_dict = {}
        if len(crystal):
            result_dict.update(crystal)
            if len(docked):
                results_dict.update(**crystal[0])
                
                

In [ ]:
groupeddf.groupby("Structure")['count'].count()

In [ ]:
fig = px.histogram(groupeddf, x="interaction_type", color="StructureType", template="simple_white")
fig.show()

In [ ]:
key, value = crystal_plint_report